# GLORYS12 — thermocline temperature time-series

Demonstrates the **depth-axis pattern** in the CMEMS backend: pull one
year of daily GLORYS12 (`thetao`, potential temperature) at three fixed
depths and plot the time-series at one ocean point.

The CMEMS toolbox returns `thetao` as a 4-D `(time, depth, lat, lon)`
NetCDF; the `minimum_depth` / `maximum_depth` kwargs let you clip
server-side so you only pay for the levels you want. This notebook
downloads the surface-to-500 m slab once and slices three target depths
client-side.

Reads credentials from `COPERNICUSMARINE_SERVICE_USERNAME` /
`COPERNICUSMARINE_SERVICE_PASSWORD` (see
[Authentication](../../reference/cmems/authentication.md)).

In [ ]:
import os
from pathlib import Path

import numpy as np

from earthlens import EarthLens
from earthlens.cmems import Catalog
from pyramids.netcdf import NetCDF

OUT_DIR = Path('data/cmems-glorys')
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_ID = 'cmems_mod_glo_phy_my_0.083deg_P1D-m'
TARGET_DEPTHS_M = (20.0, 100.0, 500.0)
POINT_LAT, POINT_LON = 35.0, -25.0           # mid-Atlantic, off the Azores
BBOX = dict(lat_lim=[34.5, 35.5], lon_lim=[-25.5, -24.5])

ds = Catalog().get_dataset(DATASET_ID)
print(f'{DATASET_ID}: cadence={ds.cadence}, domain={ds.domain}')
print('thetao units:', ds.variables['thetao'].units)

## Download — one year of daily thetao, surface to 500 m

1° x 1° box, 366 days, depths 0-500 m. The toolbox returns a NetCDF a
few MB in size.

In [ ]:
earthlens = EarthLens(
    data_source='cmems',
    start='2020-01-01',
    end='2020-12-31',
    temporal_resolution='daily',
    variables={DATASET_ID: ['thetao']},
    **BBOX,
    path=str(OUT_DIR),
    minimum_depth=0.0,
    maximum_depth=500.0,
    service_username=os.environ.get('COPERNICUSMARINE_SERVICE_USERNAME'),
    service_password=os.environ.get('COPERNICUSMARINE_SERVICE_PASSWORD'),
)
paths = earthlens.download()
print(paths)

## Slice three depths at one ocean point

Open the returned NetCDF with `pyramids.netcdf.NetCDF`, then index the
depth axis to the three target levels and the lat/lon axes to the
central pixel.

In [ ]:
nc = NetCDF.read_file(str(paths[0]), read_only=True)
thetao = nc.read_array('thetao')             # shape (time, depth, lat, lon)
depth = nc.read_array('depth')
lat = nc.read_array('latitude')
lon = nc.read_array('longitude')
time = nc.read_array('time')
print(f'thetao shape: {thetao.shape}')
print(f'depth levels in slab: {depth.tolist()}')

lat_i = int(np.argmin(np.abs(lat - POINT_LAT)))
lon_i = int(np.argmin(np.abs(lon - POINT_LON)))
depth_is = [int(np.argmin(np.abs(depth - d))) for d in TARGET_DEPTHS_M]
print(f'central pixel: lat[{lat_i}]={lat[lat_i]:.2f}, lon[{lon_i}]={lon[lon_i]:.2f}')
print(f'target depths -> depth indices: {dict(zip(TARGET_DEPTHS_M, depth_is))}')

series = {d_m: thetao[:, d_i, lat_i, lon_i] for d_m, d_i in zip(TARGET_DEPTHS_M, depth_is)}
nc.close()

## Plot the seasonal cycle at three depths

Surface follows the seasonal cycle clearly; below the thermocline (here at
100 m and 500 m) the temperature is much smoother and the seasonal signal
is damped.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
for d_m, vals in series.items():
    ax.plot(vals, label=f'{int(d_m)} m')
ax.set_xlabel('Days since 2020-01-01')
ax.set_ylabel('Potential temperature (degrees_C)')
ax.set_title('GLORYS12 thetao at 35°N, 25°W (2020)')
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()